# AquaFlow: train all models in Google Colab

This notebook trains the three AquaFlow models from scratch with the repository's own scripts:

| # | Model | Trained on | Output used by the app |
|---|---|---|---|
| A | **Leak / burst detector** | 100 simulated households, calibrated to 106 real NWSDB households, with injected leaks | `leak_model.joblib` → model service `/score` |
| B | **Monthly usage forecast** (study) | 106 real NWSDB households' monthly bills | report + `seasonality.csv` (used by C) |
| C | **Daily usage forecast** | 100 calibrated households, with the seasonality measured in B | `forecast_model.joblib` → model service `/forecast` |

**How to run:** *Runtime → Run all*, then wait. Everything is saved to Google Drive in `MyDrive/aquaflow/`.
If Colab disconnects, run all again: the finished steps are restored from Drive and skipped.

**Data honesty:** households A and C are **simulated** with SIMDEUM (Blokker et al., 2010), scaled to the
physical AquaFlow loop (pump ≈ 3.5 L/min, sensor model measured on ESP32-A1) and calibrated so their usage
matches real NWSDB bills. B uses the real (anonymised) NWSDB bills.

Approximate times on free Colab: setup 3 min · A ≈ 60–90 min (data generation is most of it) · B 2 min · C 3 min.

## 1. Settings

Change these only if you need to. `BRANCH`: use `main` once the pull request is merged.

In [ ]:
BRANCH = "claude/vibrant-cori-jjo5gr"   # or "main" after merging
REPO_URL = "https://github.com/ShewonGun/Research-Project.git"

TRAIN_LEAK = True       # A: leak / burst detector (slow: generates 100 households first)
TRAIN_MONTHLY = True    # B: monthly forecast on the real NWSDB bills
TRAIN_DAILY = True      # C: daily usage forecast
DAILY_DATA = "repo"     # "repo": use the calibrated daily totals already in the repository (fast, same data I used)
                        # "generate": regenerate them with SIMDEUM (about 1.5 GB and 1-2 extra hours)
RUN_COMPARISONS = True  # C: also compare with the current model (adds ~2 min)

N_HOUSEHOLDS = 100
DRIVE_DIR = "/content/drive/MyDrive/aquaflow"

## 2. Google Drive and the code

Allow access when Colab asks: Drive keeps the results (Colab deletes its files when the session ends).

In [ ]:
import os, sys, subprocess, shutil, time, glob
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/Research-Project")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "-q", "origin", BRANCH], check=True)
else:  # running outside Colab: use this checkout
    ROOT = Path(os.environ.get("AQUAFLOW_ROOT", Path.cwd().parents[1]))
    DRIVE_DIR = os.environ.get("AQUAFLOW_DRIVE", str(ROOT / "colab_drive"))

DRIVE = Path(DRIVE_DIR); DRIVE.mkdir(parents=True, exist_ok=True)
EDGE = ROOT / "edge-ai"
SIM = EDGE / "data" / "simdeum"
LEAK = EDGE / "leak_detection"
FC = EDGE / "usage_forecast"
NWSDB = EDGE / "data" / "nwsdb" / "nwsdb_households.csv"
print("Code:", ROOT, "| commit", subprocess.run(["git", "-C", str(ROOT), "log", "-1", "--format=%h %s"],
                                                capture_output=True, text=True).stdout.strip())
print("Drive folder:", DRIVE)

## 3. Python environment

SIMDEUM needs Python 3.11 with older pandas / numpy, so everything runs in its own Python 3.11 environment
(made with `uv`). The versions are pinned to match the model service on your PC
(numpy 1.26.4, pandas 2.1.4, scikit-learn 1.9.1, joblib 1.6.0), so the saved models load there.
You don't need to restart the runtime.

In [ ]:
VENV = Path("/content/aquaflow-env") if IN_COLAB else Path(os.environ.get("AQUAFLOW_VENV", "/tmp/aquaflow-env"))
PY = str(VENV / "bin" / "python")
PINS = ["pysimdeum==1.0.1", "numpy==1.26.4", "pandas==2.1.4", "scikit-learn==1.9.1", "joblib==1.6.0",
        "matplotlib", "holidays"]
if not Path(PY).exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "venv", "-q", "--python", "3.11", str(VENV)], check=True)
    subprocess.run(["uv", "pip", "install", "-q", "--python", PY, *PINS], check=True)
print(subprocess.run([PY, "-c", "import sys, numpy, pandas, sklearn, joblib, pysimdeum; "
                      "print('python', sys.version.split()[0], '| numpy', numpy.__version__, '| pandas', pandas.__version__, "
                      "'| scikit-learn', sklearn.__version__, '| joblib', joblib.__version__)"],
                     capture_output=True, text=True, check=True).stdout)

## 4. Helpers

`run` runs a repository script and shows its output; `generate` runs the SIMDEUM generator on all CPU
cores in parallel; `save` / `restore` keep generated data on Drive so a disconnect doesn't lose it.

In [ ]:
ENV = dict(os.environ, MPLBACKEND="Agg", PYTHONWARNINGS="ignore")


def run(script, *args, cwd):
    """Run a repository script with the training environment; stop the notebook if it fails."""
    t = time.time()
    p = subprocess.Popen([PY, script, *map(str, args)], cwd=cwd, env=ENV, text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        if "Warning" not in line:
            print(line, end="")
    if p.wait():
        raise RuntimeError(f"{script} failed (exit {p.returncode}) — see the output above")
    print(f"[{script}: {(time.time() - t) / 60:.1f} min]")


def done(out, n):
    return (out / "households.csv").exists() and len(list(out.glob("H*/1min.csv"))) >= n


def generate(name, n, *args):
    """SIMDEUM households 1..n into data/simdeum/<name>, split over the CPU cores."""
    import pandas as pd
    out = SIM / name
    if done(out, n):
        print(name, "already there"); return
    workers = max(1, min(os.cpu_count() or 1, n))
    step = -(-n // workers)
    parts = [(a, min(a + step - 1, n)) for a in range(1, n + 1, step)]
    procs = []
    for a, b in parts:
        log = open(SIM / f"{name}_{a}.log", "w")
        procs.append(subprocess.Popen([PY, "generate_simdeum_households.py", "--households", str(n), "--first", str(a),
                                       "--last", str(b), "--out", str(out), *map(str, args)],
                                      cwd=SIM, env=ENV, stdout=log, stderr=subprocess.STDOUT))
    t = time.time()
    while any(p.poll() is None for p in procs):
        time.sleep(60)
        print(f"  {name}: {len(list(out.glob('H*/1min.csv')))}/{n} households, {(time.time() - t) / 60:.0f} min", flush=True)
    for (a, _), p in zip(parts, procs):
        if p.returncode:
            print(open(SIM / f"{name}_{a}.log").read()[-3000:])
            raise RuntimeError(f"generator failed for {name} part {a}")
    files = sorted(out.glob("households_*.csv"))
    if files:
        pd.concat([pd.read_csv(f) for f in files]).sort_values("household_id").to_csv(out / "households.csv", index=False)
    print(f"{name}: {n} households in {(time.time() - t) / 60:.0f} min")


def save(path, zipname):
    """Zip a folder to Drive."""
    shutil.make_archive(str(DRIVE / zipname), "zip", root_dir=path.parent, base_dir=path.name)
    print("saved →", DRIVE / f"{zipname}.zip")


def restore(path, zipname, n):
    z = DRIVE / f"{zipname}.zip"
    if not done(path, n) and z.exists():
        shutil.unpack_archive(str(z), str(path.parent))
        print("restored", path.name, "from Drive")
    return done(path, n)

## A. Leak / burst detector

**A1. Data.** First 100 plain SIMDEUM households (1–7 Oct; only their usage levels are needed), then the
same 100 households over 1–21 Oct, **calibrated** so each one uses water as often as the real NWSDB household
at the same rank, with 4 micro-leaks and 2 bursts injected per household (labelled). Flow rates stay at loop
scale. Saved to Drive as `out_calibrated_100.zip`.

In [ ]:
if TRAIN_LEAK:
    base, cal = SIM / "out_base100", SIM / "out_calibrated_100"
    if not restore(cal, "out_calibrated_100", N_HOUSEHOLDS):
        generate("out_base100", N_HOUSEHOLDS, "--end", "2026-10-07")
        generate("out_calibrated_100", N_HOUSEHOLDS, "--calibrate-to", NWSDB, "--base", base / "households.csv")
        save(cal, "out_calibrated_100")

**A2. Train.** Household-split 6-fold cross-validation (every household is tested once, by a model that
never saw it), Random Forest vs gradient boosting vs Isolation Forest vs the backend's current rules,
ablations, then the final model with its burst and micro-leak alarms tuned to at most 0.05 false alarms
per day each. Takes about 20–40 minutes.

In [ ]:
if TRAIN_LEAK:
    run("train_leak_model.py", "--data", SIM / "out_calibrated_100", "--out", "results_calibrated_100",
        "--fa-budget", 0.05, "--final-features", "no_baseline", cwd=LEAK)

In [ ]:
from IPython.display import Markdown, Image, display
if TRAIN_LEAK:
    rep = (LEAK / "results_calibrated_100" / "REPORT.md").read_text()
    display(Markdown(rep[:6000]))
    for c in ["1_detection_vs_rules.png", "2_false_alarms_by_detector.png", "5_recall_by_shape.png"]:
        display(Image(str(LEAK / "results_calibrated_100" / "charts" / c), width=700))

## B. Monthly forecast on the real NWSDB bills

Forecasts each household's next bill from its own past bills (households and time held out). It also
measures **seasonality** (how usage changes by calendar month), which C uses. In Colab the weather
download works, so the report also says whether temperature and rain explain usage.

In [ ]:
if TRAIN_MONTHLY:
    run("train_monthly_nwsdb.py", cwd=FC)
    display(Markdown((FC / "results_monthly_nwsdb" / "REPORT.md").read_text()[:5000]))

## C. Daily usage forecast

**C1. Data.** Daily totals for 100 calibrated households (1 Jul – 9 Sep), with weekly habits, the
household's own day-to-day ups and downs and the **measured** NWSDB seasonality.
`DAILY_DATA = "repo"` uses the file already in the repository; `"generate"` makes it again from scratch.

In [ ]:
DAILY_CSV = FC / "data" / "patterned_daily_calibrated_100.csv"
if TRAIN_DAILY and DAILY_DATA == "generate":
    fcal = SIM / "out_forecast_calibrated_100"
    if not restore(fcal, "out_forecast_calibrated_100", N_HOUSEHOLDS):
        generate("out_forecast_base100", N_HOUSEHOLDS, "--start", "2026-07-01", "--end", "2026-07-07",
                 "--micro-leaks", 0, "--bursts", 0, "--seed", 7)
        generate("out_forecast_calibrated_100", N_HOUSEHOLDS, "--start", "2026-07-01", "--end", "2026-09-09",
                 "--micro-leaks", 0, "--bursts", 0, "--seed", 7, "--calibrate-to", NWSDB,
                 "--base", SIM / "out_forecast_base100" / "households.csv")
        save(fcal, "out_forecast_calibrated_100")
    season = FC / "results_monthly_nwsdb" / "seasonality.csv"
    run("make_patterned_daily.py", "--data", fcal, "--season-csv", season, "--out", DAILY_CSV, cwd=FC)
if TRAIN_DAILY:
    import pandas as pd
    d = pd.read_csv(DAILY_CSV)
    print(f"{d['home'].nunique()} households, {d['day'].min()} → {d['day'].max()}, "
          f"median {d.groupby('home')['litres'].median().median():.0f} L/day (loop scale)")

**C2. Train.** Same features and model as the deployed forecast (the next 7 days relative to the
household's own last 14 days), cross-validated by household, compared with simple rules, with an 80 % range.

In [ ]:
if TRAIN_DAILY:
    run("train_forecast.py", "--daily-csv", DAILY_CSV, "--out", "results_calibrated_100", cwd=FC)
    print(pd.read_csv(FC / "results_calibrated_100" / "forecast_metrics.csv").to_string(index=False))
    print(pd.read_csv(FC / "results_calibrated_100" / "forecast_totals.csv").to_string(index=False))
if TRAIN_DAILY and RUN_COMPARISONS:
    run("compare_forecast_calibration.py", "--calibrated", DAILY_CSV, cwd=FC)

## 5. Check the saved models

Loads each model with the pinned library versions (the same as on your PC) and makes one prediction.

In [ ]:
check = r'''
import sys, joblib, numpy as np, pandas as pd
sys.path.insert(0, sys.argv[1])
import leakdetect as ld
for path in sys.argv[2:]:
    b = joblib.load(path)
    print(path.split("edge-ai/")[-1])
    if "alarms" in b:
        df, _ = ld.load_household(__import__("pathlib").Path(sys.argv[1]).parent / "data" / "simdeum" / "out_calibrated_100" / "H01")
        p = b["model"].predict_proba(ld.make_features(df)[b["features"]].iloc[-1440:])
        print("   leak model OK:", b["model_type"], "| alarms", b["alarms"], "| classes", list(b["model"].classes_))
    else:
        X = pd.DataFrame(np.ones((1, len(b["features"]))), columns=b["features"])
        print("   forecast model OK: predicted ratio", round(float(b["model"].predict(X)[0]), 3), "| 80% half-widths",
              {k: round(v, 3) for k, v in list(b["range_halfwidth_80"].items())[:3]}, "...")
'''
paths = []
if TRAIN_LEAK: paths.append(LEAK / "results_calibrated_100" / "leak_model.joblib")
if TRAIN_DAILY: paths.append(FC / "results_calibrated_100" / "forecast_model.joblib")
if paths:
    r = subprocess.run([PY, "-c", check, str(LEAK), *map(str, paths)], capture_output=True, text=True, env=ENV)
    print(r.stdout, r.stderr[-2000:] if r.returncode else "")

## 6. Save everything to Drive

Writes `MyDrive/aquaflow/models_<date-time>/` with the two model files in the folders the service
expects, plus all reports and charts. Download that folder to your PC (see the guide).

In [ ]:
stamp = time.strftime("%Y%m%d-%H%M")
dest = DRIVE / f"models_{stamp}"
for src in [LEAK / "results_calibrated_100", FC / "results_calibrated_100", FC / "results_monthly_nwsdb",
            FC / "results_forecast_comparison"]:
    if src.exists():
        shutil.copytree(src, dest / src.relative_to(EDGE), dirs_exist_ok=True)
shutil.make_archive(str(dest), "zip", root_dir=dest)
print("Saved →", dest, "and", f"{dest}.zip")
for f in sorted(dest.rglob("*.joblib")):
    print("  model:", f.relative_to(dest), f"({f.stat().st_size / 1e6:.1f} MB)")